In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

portfolio_df = pd.read_csv(
    "../data/processed/portfolio_returns.csv",
    index_col=0,
    parse_dates=True
)

portfolio_pnl = portfolio_df["PortfolioPnL"]

PORTFOLIO_VALUE = 1_000_000

In [2]:
var_99 = np.percentile(
    portfolio_pnl,
    1
)

historical_es_99 = -portfolio_pnl[
    portfolio_pnl <= var_99
].mean()

historical_var_99 = -var_99

print(
    "Historical VaR 99%:",
    f"${historical_var_99:,.2f}"
)

print(
    "Historical ES 99%:",
    f"${historical_es_99:,.2f}"
)

Historical VaR 99%: $17,850.98
Historical ES 99%: $23,881.24


In [3]:
from scipy.stats import norm

portfolio_returns = (
    portfolio_df["PortfolioReturn"]
)

mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()

In [4]:
alpha_95 = 0.05

z_95 = norm.ppf(alpha_95)

parametric_es_95_return = -(
    mean_return
    - std_return
    * norm.pdf(z_95)
    / alpha_95
)

parametric_es_95_usd = (
    parametric_es_95_return
    * PORTFOLIO_VALUE
)

In [5]:
alpha_99 = 0.01

z_99 = norm.ppf(alpha_99)

parametric_es_99_return = -(
    mean_return
    - std_return
    * norm.pdf(z_99)
    / alpha_99
)

parametric_es_99_usd = (
    parametric_es_99_return
    * PORTFOLIO_VALUE
)

In [6]:
print(
    "Parametric ES 95%:",
    f"${parametric_es_95_usd:,.2f}"
)

print(
    "Parametric ES 99%:",
    f"${parametric_es_99_usd:,.2f}"
)

Parametric ES 95%: $14,471.01
Parametric ES 99%: $18,789.06


In [8]:
from scipy.stats import norm

portfolio_returns = portfolio_df["PortfolioReturn"]
portfolio_pnl = portfolio_df["PortfolioPnL"]

PORTFOLIO_VALUE = 1_000_000

# Historical VaR
var_95_cutoff = np.percentile(portfolio_pnl, 5)
var_99_cutoff = np.percentile(portfolio_pnl, 1)

historical_var_95 = -var_95_cutoff
historical_var_99 = -var_99_cutoff

# Historical Expected Shortfall
historical_es_95 = -portfolio_pnl[
    portfolio_pnl <= var_95_cutoff
].mean()

historical_es_99 = -portfolio_pnl[
    portfolio_pnl <= var_99_cutoff
].mean()

# Parametric inputs
mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()

# Parametric VaR
z_95 = norm.ppf(0.05)
z_99 = norm.ppf(0.01)

parametric_var_95_usd = -(
    mean_return + z_95 * std_return
) * PORTFOLIO_VALUE

parametric_var_99_usd = -(
    mean_return + z_99 * std_return
) * PORTFOLIO_VALUE

# Parametric Expected Shortfall
parametric_es_95_usd = -(
    mean_return
    - std_return * norm.pdf(z_95) / 0.05
) * PORTFOLIO_VALUE

parametric_es_99_usd = -(
    mean_return
    - std_return * norm.pdf(z_99) / 0.01
) * PORTFOLIO_VALUE

In [9]:
risk_comparison = pd.DataFrame({
    "Method": [
        "Historical",
        "Parametric"
    ],
    "VaR_95": [
        historical_var_95,
        11476.261456
    ],
    "ES_95": [
        historical_es_95,
        parametric_es_95_usd
    ],
    "VaR_99": [
        historical_var_99,
        16360.450008
    ],
    "ES_99": [
        historical_es_99,
        parametric_es_99_usd
    ]
})

risk_comparison

,Method,VaR_95,ES_95,VaR_99,ES_99
0,Historical,11656.950277,16200.951581,17850.975906,23881.241660
1,Parametric,11476.261456,14471.008711,16360.450008,18789.064976


## Expected Shortfall Interpretation

Expected Shortfall exceeded Value at Risk at both the 95% and 99% confidence levels, as expected.

Historical Expected Shortfall was materially higher than the parametric estimate, particularly at the 99% confidence level.

This suggests that the empirical return distribution contains more severe tail losses than those implied by the normal-distribution assumption used by the parametric model.

At the 99% confidence level, Historical VaR was approximately $17.9K, while Historical Expected Shortfall increased to approximately $23.9K. This indicates that once the VaR threshold is breached, average losses in the extreme tail are substantially larger than the VaR estimate itself.